# Autoresearch workflow notebook

Run the existing autoresearch loop, visualize the score trajectory, and optionally use a host-local OpenAI-compatible LLM endpoint from inside Docker.

In [ ]:
%matplotlib inline

import os
from pathlib import Path
from pprint import pprint

from autoresearch import (
    AutoresearchRunner,
    BlackjackPolicyTrainer,
    BlackjackTask,
    InventoryPolicyTrainer,
    LocalLLMResearchAgent,
    PROMPT_TEMPLATE_PRESETS,
    RestaurantInventoryTask,
    TraceableAgent,
    TrainerRegistry,
    plot_run_result,
    save_html_report,
)
from autoresearch.demo import CyclicDemoAgent


In [ ]:
TASK_NAME = os.getenv('AUTORESEARCH_NOTEBOOK_TASK', 'restaurant_inventory')
ITERATIONS = int(os.getenv('AUTORESEARCH_NOTEBOOK_ITERATIONS', '6'))
OUTPUT_ROOT = Path('artifacts/notebooks')
PROMPT_PRESET = os.getenv('AUTORESEARCH_PROMPT_PRESET', 'concise')
TEMPERATURE = float(os.getenv('AUTORESEARCH_TEMPERATURE', '0.2'))

registry = TrainerRegistry()
registry.register('restaurant_inventory', InventoryPolicyTrainer())
registry.register('blackjack', BlackjackPolicyTrainer())

if TASK_NAME == 'restaurant_inventory':
    task = RestaurantInventoryTask()
elif TASK_NAME == 'blackjack':
    task = BlackjackTask()
else:
    raise ValueError(f'Unsupported task: {TASK_NAME}')

print({'task': task.name, 'iterations': ITERATIONS, 'prompt_preset': PROMPT_PRESET})


In [ ]:
endpoint = os.getenv('AUTORESEARCH_AGENT_ENDPOINT', '').strip()
model = os.getenv('AUTORESEARCH_AGENT_MODEL', '').strip()

if endpoint and model:
    base_agent = LocalLLMResearchAgent.from_preset(
        endpoint=endpoint,
        model=model,
        prompt_preset=PROMPT_PRESET if PROMPT_PRESET in PROMPT_TEMPLATE_PRESETS else 'concise',
        temperature=TEMPERATURE,
    )
    agent_mode = 'local_llm'
else:
    base_agent = CyclicDemoAgent()
    agent_mode = 'cyclic_demo'

agent = TraceableAgent(base_agent)
print({'agent_mode': agent_mode, 'endpoint': endpoint or None, 'model': model or None})


In [ ]:
runner = AutoresearchRunner(agent=agent, registry=registry)
result = runner.run(task, iterations=ITERATIONS)

print(f'Best score: {result.best_score:.6f}')
pprint(result.best_model_state)


In [ ]:
fig = plot_run_result(result, show=False)
fig


In [ ]:
history_rows = [
    {
        'iteration': entry.iteration,
        'score': entry.score,
        'suggestion': entry.suggestion,
        'model_state': entry.model_state,
        'metrics': entry.metrics,
    }
    for entry in result.history
]
history_rows


In [ ]:
output_dir = OUTPUT_ROOT / task.name
output_dir.mkdir(parents=True, exist_ok=True)
result.to_csv(output_dir / 'results.csv')
result.to_trace_log(output_dir / 'trace.json')
save_html_report(result, output_dir / 'report.html')
print(f'Artifacts written to: {output_dir.resolve()}')
